# Convert all text in the original emails to text files. 

In [3]:
import sys
import chardet
sys.path.append('C:/Users/aaron/dev/TopicMiner/')

In [ ]:
import pandas as pd
from topicminer.utils import format_and_save_emails

In [ ]:
# Pandas display settings
pd.options.display.max_rows = 50       
pd.options.display.max_columns = 12     
pd.set_option('max_colwidth',50)        

In [ ]:
# Date
data_file = 'C:/Users/aaron/dev/TopicMiner/data/hrc/raw/data.csv'
output_dir = 'C:/Users/aaron/dev/TopicMiner/data/hrc/processed/'

In [ ]:
# Read data into a dataframe
df = pd.read_csv(data_file) 
display(df.head(4))

In [ ]:
# Format and save all the emails as text files. 
format_and_save_emails(df, output_dir)
    

In [1]:
def read_text_file(file_path: str, word_wrap_limit=100) -> list:
    """
    Read a text file and return its contents as a list of lines, automatically handling encoding detection.

    Parameters
    ----------
    file_path : str
        Path to the text file to be read.

    Returns
    -------
    list of str
        Lines read from the text file.

    Examples
    --------
    >>> file_contents = read_text_file('example.txt')
    >>> print(file_contents[0])  # Print the first line of the file.
    'This is the first line of the file.'

    Notes
    -----
    This function uses the `chardet` library to detect the encoding of the file,
    which helps in reading files with non-standard or mixed encodings.
    """
    # Uncover file encoding
    with open(file_path, "rb") as file:
        raw_data = file.read()
        encoding = chardet.detect(raw_data)["encoding"]  # Detect encoding

    # Read and return the contents of the file
    with open(file_path, "r", encoding=encoding) as file:
        text_file_contents = file.readlines()

    return text_file_contents

In [13]:
def parse_top_email_from_chain(text_file_contents: list) -> tuple:
    """
    Parses the top email from a chain in a list of lines from an Outlook formatted email text file.

    Parameters
    ----------
    text_file_contents : list of str
        Lines from a text file containing an email chain. Each line corresponds to one line of the text file.

    Returns
    -------
    tuple
        Contains 'To', 'From', 'Cc', 'Sent date', 'Subject', and 'Body' of the top email as strings.
        Each component is extracted based on Outlook format patterns.

    Notes
    -----
    This function assumes a specific format where the most recent email appears first and is followed
    by previous emails. A new email is indicated by a new "From:" line after the first.
    """
    # Initialize default values for email components
    email_recipient = "Unknown Recipient"
    email_sender = "Unknown Sender"
    email_cc = "No CC"
    email_bcc = "No BCC"
    email_date = "Unknown Date"
    email_subject = "No Subject"
    email_attachments = "No Attachments"
    email_categories = "No Categories"
    email_body = []

    first_from_found = False  # Flag to detect the first 'From:'

    # Start processing the first email
    for line in text_file_contents:
        stripped_line = line.strip()
        if stripped_line.startswith("From:"):
            if first_from_found:
                break  # Stop reading when a new email start is detected
            else:
                email_sender = stripped_line.replace("From:", "").strip()
                first_from_found = True
        elif stripped_line.startswith("Sent:"):
            email_date = stripped_line.replace("Sent:", "").strip()
        elif stripped_line.startswith("To:"):
            email_recipient = stripped_line.replace("To:", "").strip()
        elif stripped_line.startswith("Cc:"):
            email_cc = stripped_line.replace("Cc:", "").strip()
        elif stripped_line.startswith("Bcc:"):
            email_bcc = stripped_line.replace("Bcc:", "").strip()
        elif stripped_line.startswith("Subject:"):
            email_subject = stripped_line.replace("Subject:", "").strip()
        elif stripped_line.startswith("Attachments:"):
            email_attachments = stripped_line.replace("Attachments:", "").strip()
        elif stripped_line.startswith("Categories:"):
            email_categories = stripped_line.replace("Categories:", "").strip()
        elif stripped_line:
            email_body.append(stripped_line)  # Collecting body text

    email_body = "\n".join(email_body)  # Join all body lines into a single string
    return (
        email_recipient,
        email_sender,
        email_cc,
        email_bcc,
        email_date,
        email_subject,
        email_attachments,
        email_categories,
        email_body,
    )




In [8]:
import re

In [37]:
text_file_contents = read_text_file(file_path = '/home/aaronnhorvitz/dev/work/irs/TopicMiner/data/fake_data_generator/emails/00710.txt')
text_file_contents

['\n',
 'From:    Donna Clark\n',
 'Sent:    Wednesday, February 02, 1983 02:51 PM\n',
 'To:      Donna Clark\n',
 '\n',
 'Categories:    Meeting Request\n',
 '\n',
 'Dear Gordon,\n',
 '\n',
 "I'm thrilled to hear that you're willing to lend your expertise in optimizing our testing procedures. Your input would be invaluable in helping us catch potential roadblocks early on.\n",
 '\n',
 "I've been thinking a lot about our goals and timelines, and I believe we could improve our processes by implementing a more thorough QA phase upfront. Your experience in this area would be incredibly helpful in making sure we're covering all the bases.\n",
 '\n',
 "Let's schedule that meeting to discuss our project updates and potential roadblocks. I'm looking forward to hearing your thoughts on how we can optimize our testing procedures.\n",
 '\n',
 'Very truly yours,\n',
 'Donna Clark\n',
 '+1 512-555-0156\n',
 'Director of Operations, Cardiff Electric, Operations\n',
 '\n',
 '\n',
 'From:    Gordon C

In [38]:

(
    email_recipient,email_sender,
    email_cc,
    email_bcc,
    email_date,
    email_subject,
    email_attachments,
    email_categories,
    email_body) =parse_top_email_from_chain(text_file_contents)


In [39]:
(
    email_recipient,email_sender,
    email_cc,
    email_bcc,
    email_date,
    email_subject,
    email_attachments,
    email_categories,
    email_body) =parse_top_email_from_chain(text_file_contents)


In [40]:
print(email_sender)

Donna Clark


In [41]:
print(email_body)

Dear Gordon,
I'm thrilled to hear that you're willing to lend your expertise in optimizing our testing procedures. Your input would be invaluable in helping us catch potential roadblocks early on.
I've been thinking a lot about our goals and timelines, and I believe we could improve our processes by implementing a more thorough QA phase upfront. Your experience in this area would be incredibly helpful in making sure we're covering all the bases.
Let's schedule that meeting to discuss our project updates and potential roadblocks. I'm looking forward to hearing your thoughts on how we can optimize our testing procedures.
Very truly yours,
Donna Clark
+1 512-555-0156
Director of Operations, Cardiff Electric, Operations


In [61]:
import json
UNWANTED_TEXTS_FILE = '/home/aaronnhorvitz/dev/work/irs/TopicMiner/data/unwanted_texts/unwanted_texts.json'

In [62]:

def load_unwanted_email_text():

    try:
        with open(UNWANTED_TEXTS_FILE, "r") as file:
            data = json.load(file)
        if "unwanted_texts" in data:
            return set(data["unwanted_texts"])
        else:
            raise ValueError(
                f"JSON file at {UNWANTED_TEXTS_FILE} is missing the 'unwanted_texts' key."
            )
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error reading JSON file: {e}")
        return set()
    except ValueError as ve:
        print(ve)
        return set()

In [66]:
file_path = '/home/aaronnhorvitz/dev/work/irs/TopicMiner/data/fake_data_generator/emails/00710.txt'
unwanted_texts = load_unwanted_email_text()

with open(file_path, "r", encoding="utf-8") as file:
    email_content = file.read()

In [67]:
email_content

"\nFrom:    Donna Clark\nSent:    Wednesday, February 02, 1983 02:51 PM\nTo:      Donna Clark\n\nCategories:    Meeting Request\n\nDear Gordon,\n\nI'm thrilled to hear that you're willing to lend your expertise in optimizing our testing procedures. Your input would be invaluable in helping us catch potential roadblocks early on.\n\nI've been thinking a lot about our goals and timelines, and I believe we could improve our processes by implementing a more thorough QA phase upfront. Your experience in this area would be incredibly helpful in making sure we're covering all the bases.\n\nLet's schedule that meeting to discuss our project updates and potential roadblocks. I'm looking forward to hearing your thoughts on how we can optimize our testing procedures.\n\nVery truly yours,\nDonna Clark\n+1 512-555-0156\nDirector of Operations, Cardiff Electric, Operations\n\n\nFrom:    Gordon Clark\nSent:    Wednesday, February 02, 1983 02:51 PM\nTo:      Donna Clark\n\nCategories:    Meeting Reque

In [68]:


# Process the email content to remove unwanted texts
screened_email_content = email_content
for phrase in unwanted_texts:
    screened_email_content = screened_email_content.replace(phrase, "")


In [70]:
screened_email_content.splitlines()

['',
 'From:    Donna Clark',
 'Sent:    Wednesday, February 02, 1983 02:51 PM',
 'To:      Donna Clark',
 '',
 'Categories:    Meeting Request',
 '',
 'Dear Gordon,',
 '',
 "I'm thrilled to hear that you're willing to lend your expertise in optimizing our testing procedures. Your input would be invaluable in helping us catch potential roadblocks early on.",
 '',
 "I've been thinking a lot about our goals and timelines, and I believe we could improve our processes by implementing a more thorough QA phase upfront. Your experience in this area would be incredibly helpful in making sure we're covering all the bases.",
 '',
 "Let's schedule that meeting to discuss our project updates and potential roadblocks. I'm looking forward to hearing your thoughts on how we can optimize our testing procedures.",
 '',
 'Very truly yours,',
 'Donna Clark',
 '+1 512-555-0156',
 'Director of Operations, Cardiff Electric, Operations',
 '',
 '',
 'From:    Gordon Clark',
 'Sent:    Wednesday, February 02, 

In [57]:

# Strip top email
(
    email_recipient,
    email_sender,
    email_cc,
    email_bcc,
    email_date,
    email_subject,
    email_attachments,
    email_categories,
    email_body
    ) = parse_top_email_from_chain(screened_email_content)

In [59]:
email_recipient

'Unknown Recipient'

In [ ]:
def parse_top_email_from_chain(text_file_contents: list) -> tuple:
    """
    Parses the top email from a chain in a list of lines from an Outlook formatted email text file.

    Parameters
    ----------
    text_file_contents : list of str
        Lines from a text file containing an email chain. Each line corresponds to one line of the text file.

    Returns
    -------
    tuple
        Contains 'To', 'From', 'Cc', 'Sent date', 'Subject', and 'Body' of the top email as strings.
        Each component is extracted based on Outlook format patterns.

    Notes
    -----
    This function assumes a specific format where the most recent email appears first and is followed
    by previous emails. A new email is indicated by a new "From:" line after the first.
    """
    # Initialize default values for email components
    email_recipient = "Unknown Recipient"
    email_sender = "Unknown Sender"
    email_cc = "No CC"
    email_bcc = "No BCC"
    email_date = "Unknown Date"
    email_subject = "No Subject"
    email_attachments = "No Attachments"
    email_categories = "No Categories"
    email_body = []

    first_from_found = False  # Flag to detect the first 'From:'

    # Start processing the first email
    for line in text_file_contents:
        stripped_line = line.strip()
        if stripped_line.startswith("From:"):
            if first_from_found:
                break  # Stop reading when a new email start is detected
            else:
                email_sender = stripped_line.replace("From:", "").strip()
                first_from_found = True
        elif stripped_line.startswith("Sent:"):
            email_date = stripped_line.replace("Sent:", "").strip()
        elif stripped_line.startswith("To:"):
            email_recipient = stripped_line.replace("To:", "").strip()
        elif stripped_line.startswith("Cc:"):
            email_cc = stripped_line.replace("Cc:", "").strip()
        elif stripped_line.startswith("Bcc:"):
            email_bcc = stripped_line.replace("Bcc:", "").strip()
        elif stripped_line.startswith("Subject:"):
            email_subject = stripped_line.replace("Subject:", "").strip()
        elif stripped_line.startswith("Attachments:"):
            email_attachments = stripped_line.replace("Attachments:", "").strip()
        elif stripped_line.startswith("Categories:"):
            email_categories = stripped_line.replace("Categories:", "").strip()
        elif stripped_line:
            email_body.append(stripped_line)  # Collecting body text

    email_body = "\n".join(email_body)  # Join all body lines into a single string
    return (
        email_recipient,
        email_sender,
        email_cc,
        email_bcc,
        email_date,
        email_subject,
        email_attachments,
        email_categories,
        email_body,
    )




In [36]:
email_body

"F\nr\no\nm\n:\nD\no\nn\nn\na\nC\nl\na\nr\nk\nS\ne\nn\nt\n:\nW\ne\nd\nn\ne\ns\nd\na\ny\n,\nF\ne\nb\nr\nu\na\nr\ny\n0\n2\n,\n1\n9\n8\n3\n0\n2\n:\n5\n1\nP\nM\nT\no\n:\nD\no\nn\nn\na\nC\nl\na\nr\nk\nC\na\nt\ne\ng\no\nr\ni\ne\ns\n:\nM\ne\ne\nt\ni\nn\ng\nR\ne\nq\nu\ne\ns\nt\nD\ne\na\nr\nG\no\nr\nd\no\nn\n,\nI\n'\nm\nt\nh\nr\ni\nl\nl\ne\nd\nt\no\nh\ne\na\nr\nt\nh\na\nt\ny\no\nu\n'\nr\ne\nw\ni\nl\nl\ni\nn\ng\nt\no\nl\ne\nn\nd\ny\no\nu\nr\ne\nx\np\ne\nr\nt\ni\ns\ne\ni\nn\no\np\nt\ni\nm\ni\nz\ni\nn\ng\no\nu\nr\nt\ne\ns\nt\ni\nn\ng\np\nr\no\nc\ne\nd\nu\nr\ne\ns\n.\nY\no\nu\nr\ni\nn\np\nu\nt\nw\no\nu\nl\nd\nb\ne\ni\nn\nv\na\nl\nu\na\nb\nl\ne\ni\nn\nh\ne\nl\np\ni\nn\ng\nu\ns\nc\na\nt\nc\nh\np\no\nt\ne\nn\nt\ni\na\nl\nr\no\na\nd\nb\nl\no\nc\nk\ns\ne\na\nr\nl\ny\no\nn\n.\nI\n'\nv\ne\nb\ne\ne\nn\nt\nh\ni\nn\nk\ni\nn\ng\na\nl\no\nt\na\nb\no\nu\nt\no\nu\nr\ng\no\na\nl\ns\na\nn\nd\nt\ni\nm\ne\nl\ni\nn\ne\ns\n,\na\nn\nd\nI\nb\ne\nl\ni\ne\nv\ne\nw\ne\nc\no\nu\nl\nd\ni\nm\np\nr\no\nv\ne\no\nu\nr\np\nr\no\n

In [72]:
import pandas as pd
json_file_path = '/home/aaronnhorvitz/dev/work/irs/TopicMiner/data/processed_data/json_format/processed_emails.json'

# Load data
with open(json_file_path, 'r') as file:
    emails = json.load(file)

# Convert to DataFrame
df_emails = pd.DataFrame(emails)

In [73]:
df_emails

,doc_id,date,email_recipient,email_sender,email_cc,email_bcc,email_subject,email_attachments,email_categories,email_body,processed_text,word_count,character_count,token_count,file_path
0,ce8cacfd7f9571507f0ffbee0de5baa24b7d47f6f54067...,"Tuesday, January 18, 1983 03:45 PM",Gordon Clark; Debbie Malinowski,Yo-Yo Engberk,No CC,No BCC,Life's Too Short to Code Alone,No Attachments,Personal Update,"Hello Gordon and Debbie,\nHey, team! It's been...",life too short code alone hello gordon debbie ...,116,652,61,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
1,93db29dae80122acf3fc7aa95455d29fc5e0489de45d93...,"Tuesday, January 04, 1983 09:54 AM",John Bosworth; Yo-Yo Engberk; Joe MacMillan; G...,Debbie Malinowski,No CC,No BCC,Industry News Update,No Attachments,Industry News,"Dear All,\nI hope this finds you well. It's ha...",industry news update dear all hope find well i...,126,809,76,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
2,1ce49600bc3901c2de70bf61ce988443aeb3b9ccaac97d...,"Thursday, January 13, 1983 11:27 AM",Debbie Malinowski,Ed Burris,No CC,No BCC,Social Event Planning,No Attachments,Social Event,"Dear Debbie,\nI'm excited to initiate planning...",social event planning dear debbie excited init...,135,882,87,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
3,c0792c5c8967322e3bdf0dc4afd8935f2478bfa08bc7dc...,"Tuesday, January 25, 1983 03:10 PM",Malcolm Levitan; Barry Shields; Ed Burris; Joh...,Yo-Yo Engberk,No CC,No BCC,Code Crusade: Training and Development,No Attachments,Training and Development,"Hi Team,\nI hope this email finds you all fuel...",code crusade training development hi team hope...,145,856,82,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
4,fc4a30b50e9bd96a6d9b92aaa7026be3835da52774cc78...,"Monday, January 17, 1983 09:42 AM",Malcolm Levitan,Yo-Yo Engberk,No CC,No BCC,,No Attachments,Technical Support,"Bug Bites and Tech Troubles\nDear Malcolm,\nI'...",bug bites tech troubles dear malcolm writing t...,110,628,62,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,ceb4bac5a6c59eab94f94fef5dee74827dfc2ba1015cb4...,"Monday, January 10, 1983 03:33 PM",Joe MacMillan,Barry Shields,No CC,No BCC,Social Event: Formal Invitation and Guidelines,No Attachments,Social Event,"Dear Joe,\nI am compelled to write this email ...",social event formal invitation guidelines dear...,174,1164,104,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
498,4b558155ec198a920f3890ed1efc33d9372cedc31c8dcf...,"Monday, January 24, 1983 01:33 PM",Joe MacMillan; Malcolm Levitan; Donna Clark; G...,Debbie Malinowski,No CC,No BCC,Reminder,No Attachments,Reminder,"Hello Team,\nI'm excited to share with you tha...",reminder hello team excited share quarterly go...,70,436,40,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
499,9a61f5af26e403af0b654311c49eea3142714184b0ca47...,"Wednesday, January 19, 1983 01:05 PM",Yo-Yo Engberk; Ed Burris; Debbie Malinowski,John Bosworth,No CC,No BCC,Meeting Request,No Attachments,Meeting Request,"Hello Yo-Yo, Ed, and Debbie,\nI'm reaching out...",meeting request hello ed debbie reaching sched...,104,599,52,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
500,bc2a8fab701bec351e52785117c4f479fe04ccfa38337a...,"Wednesday, January 05, 1983 11:47 AM",Malcolm Levitan,Larry Goins,No CC,No BCC,Compliance Inquiry,No Attachments,Compliance,"Hello Malcolm,\nI'm thrilled to be assisting y...",compliance inquiry hello malcolm thrilled assi...,158,1003,92,/home/aaronnhorvitz/dev/work/irs/TopicMiner/da...
